In [ ]:
import pandas as pd
import duckdb

# make sure to replace 'file.parquet' with your file path
df = pd.read_parquet('./data/idMinMaxDepth.parquet')
# df = df.head(10)

# use the values in the dataset_id to search via
# duckdb for the @id to generate the JSON-LD with.

def search_duckdb(x):
    x = duckdb.sql(f"SELECT url FROM read_json('./jsonld/obis_source/*.jsonld') WHERE url like '%{x}%'").fetchall()
    xs = [str(item[0]) for item in x]
    return(xs)

df['docid'] = df['dataset_id'].apply(lambda x: search_duckdb(x))

dfe = df.explode('docid')


In [4]:
dfe

,dataset_id,Max(maximumDepthInMeters),Min(minimumDepthInMeters),docid
0,7980442b-9f1f-416a-8c61-d58b03a7a61e,NaN,0.0,https://obis.org/dataset/7980442b-9f1f-416a-8c...
0,7980442b-9f1f-416a-8c61-d58b03a7a61e,NaN,0.0,https://obis.org/dataset/7980442b-9f1f-416a-8c...
0,7980442b-9f1f-416a-8c61-d58b03a7a61e,NaN,0.0,https://obis.org/dataset/7980442b-9f1f-416a-8c...
1,a9a3bdc6-209f-4c66-aafd-ce5271cb63b3,1411.0,-55.0,https://obis.org/dataset/a9a3bdc6-209f-4c66-aa...
1,a9a3bdc6-209f-4c66-aafd-ce5271cb63b3,1411.0,-55.0,https://obis.org/dataset/a9a3bdc6-209f-4c66-aa...
...,...,...,...,...
4873,3c59e9b2-15d1-4b40-80b0-f08d8293d551,NaN,NaN,https://obis.org/dataset/3c59e9b2-15d1-4b40-80...
4873,3c59e9b2-15d1-4b40-80b0-f08d8293d551,NaN,NaN,https://obis.org/dataset/3c59e9b2-15d1-4b40-80...
4874,9080ddf4-075c-4bf1-875c-338998c7f897,NaN,NaN,https://obis.org/dataset/9080ddf4-075c-4bf1-87...
4874,9080ddf4-075c-4bf1-875c-338998c7f897,NaN,NaN,https://obis.org/dataset/9080ddf4-075c-4bf1-87...


In [7]:
# remove where columns min/max have Nan
dfe_strict = dfe.dropna(subset=['Max(maximumDepthInMeters)', 'Min(minimumDepthInMeters)'], how='any')



In [8]:
dfe_strict

,dataset_id,Max(maximumDepthInMeters),Min(minimumDepthInMeters),docid,jsonld
1,a9a3bdc6-209f-4c66-aafd-ce5271cb63b3,1411.0,-55.000000,https://obis.org/dataset/a9a3bdc6-209f-4c66-aa...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
1,a9a3bdc6-209f-4c66-aafd-ce5271cb63b3,1411.0,-55.000000,https://obis.org/dataset/a9a3bdc6-209f-4c66-aa...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
1,a9a3bdc6-209f-4c66-aafd-ce5271cb63b3,1411.0,-55.000000,https://obis.org/dataset/a9a3bdc6-209f-4c66-aa...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
1,a9a3bdc6-209f-4c66-aafd-ce5271cb63b3,1411.0,-55.000000,https://obis.org/dataset/a9a3bdc6-209f-4c66-aa...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
2,9c2d6787-245d-4661-b052-c14bccfe1c1b,15.6,-32.900002,https://obis.org/dataset/9c2d6787-245d-4661-b0...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
...,...,...,...,...,...
4854,2786a328-5d83-458d-ba7f-96e055f4432e,0.0,0.000000,https://obis.org/dataset/2786a328-5d83-458d-ba...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
4856,fb7d3788-7747-442c-ab9e-2cfeb4923031,7.0,5.000000,https://obis.org/dataset/fb7d3788-7747-442c-ab...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
4870,f7389969-2a16-484d-8dfe-ecf1b6f36d4b,15.0,4.000000,https://obis.org/dataset/f7389969-2a16-484d-8d...,"{\n ""@context"": {\n ""@vocab"": ""ht..."
4870,f7389969-2a16-484d-8dfe-ecf1b6f36d4b,15.0,4.000000,https://obis.org/dataset/f7389969-2a16-484d-8d...,"{\n ""@context"": {\n ""@vocab"": ""ht..."


In [2]:
def populate_template(row):
    template = """ {{
      "@context": {{
        "@vocab": "https://schema.org/"
      }},
      "@id": "{docid}",
      "@type": "Dataset",
      "variableMeasured": [
        {{
          "@type": "PropertyValue",
          "name": "depth",
          "description": "Parsed and validated by OBIS.",
          "minValue": "{MIN}",
          "maxValue": "{MAX}",
          "propertyID": "https://obis.org/data/access/",
          "measurementTechnique": "Parsed and validated by OBIS.",
          "unitText": "m",
          "unitCode": [
            "https://qudt.org/vocab/unit/M", "https://vocab.nerc.ac.uk/collection/P06/current/ULAA/",
            "http://dbpedia.org/resource/Metre"
          ]
        }}
      ]
    }}
    """
    
    return template.format(MAX=row['Max(maximumDepthInMeters)'], MIN=row['Min(minimumDepthInMeters)'],  docid=row['docid'])




In [3]:
dfe = dfe.assign(jsonld=dfe.apply(populate_template, axis=1))


In [5]:

for index, row in dfe.iterrows():
    filename = str('./jsonld/output_raw/' + row['dataset_id']) + '_depth.jsonld'  # adjust file extension as per your requirement
    with open(filename, 'w') as f:
        f.write(row['jsonld'])

In [10]:
dfe_strict = dfe_strict.assign(jsonld=dfe_strict.apply(populate_template, axis=1))


In [11]:
for index, row in dfe_strict.iterrows():
    filename = str('./jsonld/output/' + row['dataset_id']) + '_depth.jsonld'  # adjust file extension as per your requirement
    with open(filename, 'w') as f:
        f.write(row['jsonld'])